In [1]:
import pandas as pd
import sqlite3

# Connect to the database we created in Phase 1
conn = sqlite3.connect("../data/churn.db")
print("✓ Connected to churn.db")

✓ Connected to churn.db


In [2]:
# ── Query 1: Overall Churn Rate ──────────────────────────────
# Overall churn rate for database

query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(churn) AS total_churned,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 1) AS churn_rate_pct
FROM customers
"""

pd.read_sql(query, conn)

,total_customers,total_churned,churn_rate_pct
0,7043,1869,26.5


In [3]:
# ── Query 2: Churn by Contract Type ─────────────────────────
# Month-to-month customers churn far more than long-term contracts

query = """
SELECT
    contract,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY contract
ORDER BY churn_rate_pct DESC
"""

pd.read_sql(query, conn)

,contract,total_customers,churned,churn_rate_pct
0,Month-to-month,3875,1655,42.7
1,One year,1473,166,11.3
2,Two year,1695,48,2.8


In [4]:
# ── Query 3: Churn by Tenure Bucket ─────────────────────────
# New customers (0-12 months) churn the most — key insight!

query = """
SELECT
    CASE
        WHEN tenure BETWEEN 0  AND 12 THEN '0-12 months'
        WHEN tenure BETWEEN 13 AND 24 THEN '13-24 months'
        WHEN tenure BETWEEN 25 AND 48 THEN '25-48 months'
        WHEN tenure BETWEEN 49 AND 72 THEN '49-72 months'
    END   AS tenure_bucket,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY tenure_bucket
ORDER BY churn_rate_pct DESC
"""

pd.read_sql(query, conn)

,tenure_bucket,total_customers,churned,churn_rate_pct
0,0-12 months,2186,1037,47.4
1,13-24 months,1024,294,28.7
2,25-48 months,1594,325,20.4
3,49-72 months,2239,213,9.5


In [5]:
# ── Query 4: Churn by Payment Method ────────────────────────
# Electronic check users churn the most

query = """
SELECT
    paymentmethod,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY paymentmethod
ORDER BY churn_rate_pct DESC
"""

pd.read_sql(query, conn)

,paymentmethod,total_customers,churned,churn_rate_pct
0,Electronic check,2365,1071,45.3
1,Mailed check,1612,308,19.1
2,Bank transfer (automatic),1544,258,16.7
3,Credit card (automatic),1522,232,15.2


In [6]:
# ── Query 5: Revenue at Risk ─────────────────────────────────
# Translates churn into a real business dollar impact

query = """
SELECT
    ROUND(SUM(monthlycharges), 2) AS total_monthly_revenue,
    ROUND(SUM(CASE WHEN churn = 1
              THEN monthlycharges ELSE 0 END), 2) AS revenue_lost_to_churn,
    ROUND(SUM(CASE WHEN churn = 1
              THEN monthlycharges ELSE 0 END)
          * 100.0 / SUM(monthlycharges), 1) AS pct_revenue_at_risk
FROM customers
"""

pd.read_sql(query, conn)

,total_monthly_revenue,revenue_lost_to_churn,pct_revenue_at_risk
0,456116.6,139130.85,30.5


In [7]:
# ── Query 6: Churn by Internet Service ──────────────────────
# Fiber optic customers pay more but churn more too

query = """
SELECT
    internetservice,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned,
    ROUND(AVG(monthlycharges), 2) AS avg_monthly_charges,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY internetservice
ORDER BY churn_rate_pct DESC
"""

pd.read_sql(query, conn)

,internetservice,total_customers,churned,avg_monthly_charges,churn_rate_pct
0,Fiber optic,3096,1297,91.50,41.9
1,DSL,2421,459,58.10,19.0
2,No,1526,113,21.08,7.4


In [8]:
# ── Query 7: Window Function — Spend Rank of Churned Customers
# Demonstrating use of window functions

query = """
SELECT
    customerid,
    monthlycharges,
    tenure,
    contract,
    RANK() OVER (ORDER BY monthlycharges DESC) AS spend_rank,
    NTILE(4) OVER (ORDER BY monthlycharges) AS spend_quartile
FROM customers
WHERE churn = 1
ORDER BY monthlycharges DESC
LIMIT 20
"""

pd.read_sql(query, conn)

,customerid,monthlycharges,tenure,contract,spend_rank,spend_quartile
0,8199-ZLLSA,118.35,67,One year,1,4
1,2889-FPWRM,117.80,72,One year,2,4
2,2302-ANTDP,117.45,48,Month-to-month,3,4
3,9053-JZFKV,116.20,67,Two year,4,4
4,1444-VVSGW,115.65,70,One year,5,4
5,0201-OAMXR,115.55,70,One year,6,4
6,4361-BKAXE,114.50,41,Month-to-month,7,4
7,1555-DJEQW,114.20,70,Two year,8,4
8,9158-VCTQB,113.60,41,Month-to-month,9,4
9,7279-BUYWN,113.20,41,Month-to-month,10,4


In [9]:
# ── Query 8: High-Value Churned Customers (CTE) ──────────────
# Who were the most valuable customers we lost?

query = """
WITH high_value AS (
    SELECT
        customerid,
        monthlycharges,
        totalcharges,
        tenure,
        contract,
        paymentmethod
    FROM customers
    WHERE churn = 1
      AND monthlycharges > (SELECT AVG(monthlycharges) FROM customers)
)
SELECT
    COUNT(*) AS high_value_churned,
    ROUND(AVG(monthlycharges), 2) AS avg_monthly_charges,
    ROUND(AVG(tenure), 1) AS avg_tenure_months,
    ROUND(SUM(monthlycharges), 2) AS total_monthly_revenue_lost
FROM high_value
"""

pd.read_sql(query, conn)

,high_value_churned,avg_monthly_charges,avg_tenure_months,total_monthly_revenue_lost
0,1355,87.5,20.6,118560.0


In [10]:
# ── Query 9: Churn Risk Matrix — Tenure x Contract ───────────
# For Tableau heatmap

query = """
SELECT
    CASE
        WHEN tenure BETWEEN 0  AND 12 THEN '0-12 months'
        WHEN tenure BETWEEN 13 AND 24 THEN '13-24 months'
        WHEN tenure BETWEEN 25 AND 48 THEN '25-48 months'
        WHEN tenure BETWEEN 49 AND 72 THEN '49-72 months'
    END AS tenure_bucket,
    contract,
    COUNT(*) AS total_customers,
    SUM(churn) AS churned,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY tenure_bucket, contract
ORDER BY churn_rate_pct DESC
"""

pd.read_sql(query, conn)

,tenure_bucket,contract,total_customers,churned,churn_rate_pct
0,0-12 months,Month-to-month,1994,1024,51.4
1,13-24 months,Month-to-month,737,278,37.7
2,25-48 months,Month-to-month,802,264,32.9
3,49-72 months,Month-to-month,342,89,26.0
4,49-72 months,One year,634,82,12.9
5,25-48 months,One year,518,55,10.6
6,0-12 months,One year,124,13,10.5
7,13-24 months,One year,197,16,8.1
8,49-72 months,Two year,1263,42,3.3
9,25-48 months,Two year,274,6,2.2


In [12]:
# Close database connection at the end
conn.close()
print("✓ Connection closed")

✓ Connection closed
